# Video Streaming: Feature Extraction from Video Streams

In this assignment, you will explore a capture of a Netflix video stream. The packet capture itself has some additional traffic beyond Netflix traffic, and so part of the exercise involves filtering the traffic to include only the Netflix traffic.

## Learning Objectives

In this hands-on activity, you will learn how to:

* Identify service types using DNS
* Calculate network counters
* Infer video segment downloads


## Step 1: Identifying Netflix Traffic from DNS

One of the challenges with packet captures is that they often contain a mix of traffic from devices, destinations, and applications. When diagnosing performance problems with a particular service, often the first challenge is identifying and extracting the subset of traffic corresponding to that service.

In this exercise, we will use the domain name system lookups to a set of domains that we know are associated with Netflix to identify the IP addresses (and thus, the traffic flows) that are associated with Netflix.

In [27]:
import dnslib

NF_DOMAINS = (["nflxvideo", 
              "netflix", 
              "nflxso", 
              "nflxext"])

### Load the Packet Capture and Identify Netflix Traffic

First, load the traffic capture and inspect it.

In [28]:
import pandas as pd

ndf = pd.read_csv("data/netflix.csv.gz")
ndf.head(20)

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,2018-02-11 08:10:00.534682,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0xed0c A fonts.gstatic.com
1,2,2018-02-11 08:10:00.534832,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0x301a AAAA fonts.gstatic.com
2,3,2018-02-11 08:10:00.539408,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x11d3 A googleads.g.doubleclic...
3,4,2018-02-11 08:10:00.541204,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x1284 AAAA googleads.g.doublec...
4,5,2018-02-11 08:10:00.545785,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,78,Standard query 0x3432 AAAA ytimg.l.google.com
5,6,2018-02-11 08:10:00.547036,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,96,Standard query 0xb756 A r4---sn-gxo5uxg-jqbe.g...
6,7,2018-02-11 08:10:00.547156,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x62ab A ssl.gstatic.com
7,8,2018-02-11 08:10:00.547249,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,74,Standard query 0x42fb A www.google.com
8,9,2018-02-11 08:10:00.853950,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,386,Standard query response 0x11d3 A 216.58.213.162
9,10,2018-02-11 08:10:00.853970,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x8756 A www.gstatic.com


#### Filter for DNS Traffic 

Next, write an expression that filters the dataframe to include only DNS traffic.

In [29]:
dns_traffic = ndf[ndf['Protocol'] == 'DNS']
print(f"There are a total of {len(dns_traffic)} DNS packets.")

There are a total of 100 DNS packets.


#### Filter for Netflix DNS Query IDs

Because you are looking for the IP addresses that are associated with Netflix traffic, you need to match the *responses* of the corresponding DNS lookups to the queries that contain Netflix domains. You can link them with the transaction ID in the DNS traffic.

In [30]:
# Netflix DNS *queries*: the domain name only appears in the query, not the response.
nf_queries = dns_traffic[
    dns_traffic['Info'].str.contains('Standard query 0x', na=False)
    & dns_traffic['Info'].str.contains('|'.join(NF_DOMAINS), case=False, na=False)
]

# The transaction ID is the 0x.... token in the Info string.
nf_query_ids = set(nf_queries['Info'].str.extract(r'query (0x[0-9a-f]{4})')[0])

# Keep the responses whose transaction ID matches one of the Netflix queries.
responses = dns_traffic[dns_traffic['Info'].str.contains('query response', na=False)]
response_ids = responses['Info'].str.extract(r'response (0x[0-9a-f]{4})')[0]

nf_responses = responses[response_ids.isin(nf_query_ids)]
nf_responses[['Time', 'Info']]


,Time,Info
101,2018-02-11 08:10:02.902769,Standard query response 0x7776 A 198.38.120.130
102,2018-02-11 08:10:02.902776,Standard query response 0xb19a A 52.19.39.146
103,2018-02-11 08:10:02.902810,Standard query response 0x1f03 A 52.210.19.176
104,2018-02-11 08:10:02.902831,Standard query response 0x37e9 A 198.38.120.153
105,2018-02-11 08:10:02.902858,Standard query response 0x3049 A 23.57.80.120
108,2018-02-11 08:10:02.903312,Standard query response 0x5f60 A 23.57.80.120
219,2018-02-11 08:10:03.642910,Standard query response 0x6677 A 198.38.120.137
228,2018-02-11 08:10:03.670254,Standard query response 0x9415 A 198.38.120.167
1019,2018-02-11 08:10:12.323453,Standard query response 0x5542 A 34.252.77.54
1566,2018-02-11 08:10:20.775688,Standard query response 0x269f A 198.38.120.134


#### Find the Associated Netflix IP Addresses

Get the IP addresses associated with all Netflix traffic in the trace.

In [31]:
# Map each transaction ID to the domain it asked about, and to the IP it got back.
qid_to_domain = dict(zip(
    nf_queries['Info'].str.extract(r'query (0x[0-9a-f]{4})')[0],
    nf_queries['Info'].str.extract(r'query 0x[0-9a-f]{4} A+ (\S+)$')[0],
))
qid_to_ip = dict(zip(
    nf_responses['Info'].str.extract(r'response (0x[0-9a-f]{4})')[0],
    nf_responses['Info'].str.extract(r' A (\d+\.\d+\.\d+\.\d+)$')[0],
))

# Join on the transaction ID to get domain -> IP addresses.
nf_domain_ips = {}
for qid, ip in qid_to_ip.items():
    nf_domain_ips.setdefault(qid_to_domain[qid], set()).add(ip)

nf_ips = set(qid_to_ip.values())

for domain in sorted(nf_domain_ips):
    print(f"{domain:45s} {sorted(nf_domain_ips[domain])}")
print(f"\n{len(nf_domain_ips)} domains, {len(nf_ips)} unique IP addresses")


assets.nflxext.com                            ['23.57.80.120']
codex.nflxext.com                             ['23.57.80.120']
customerevents.netflix.com                    ['52.210.19.176']
ipv4-c001-cdg001-ix.1.oca.nflxvideo.net       ['198.38.120.130']
ipv4-c005-cdg001-ix.1.oca.nflxvideo.net       ['198.38.120.134']
ipv4-c024-cdg001-ix.1.oca.nflxvideo.net       ['198.38.120.153']
ipv4-c063-cdg001-ix.1.oca.nflxvideo.net       ['198.38.120.162']
ipv4-c069-cdg001-ix.1.oca.nflxvideo.net       ['198.38.120.164']
ipv4-c071-cdg001-ix.1.oca.nflxvideo.net       ['198.38.120.166']
ipv4-c072-cdg001-ix.1.oca.nflxvideo.net       ['198.38.120.167']
occ-0-56-55.1.nflxso.net                      ['198.38.120.137']
push.prod.netflix.com                         ['34.252.77.54']
www.netflix.com                               ['52.19.39.146', '52.208.128.101', '52.210.133.255', '52.48.148.78', '52.48.8.150']

13 domains, 16 unique IP addresses


In [32]:
# Heads up: this capture was exported with Wireshark name resolution on, so the
# busiest hosts appear in Source/Destination as names, not as the IPs above.
# Match on either form, or you lose ~99% of the video traffic.
endpoints = pd.concat([ndf['Source'], ndf['Destination']]).unique()
nf_hostnames = {
    e for e in endpoints
    if isinstance(e, str) and any(d in e.lower() for d in NF_DOMAINS)
}
nf_endpoints = nf_ips | nf_hostnames

netflix_traffic = ndf[ndf['Source'].isin(nf_endpoints) | ndf['Destination'].isin(nf_endpoints)]
print(f"{len(nf_endpoints)} Netflix endpoints")
print(f"{len(netflix_traffic)} of {len(ndf)} packets, {netflix_traffic['Length'].sum():,} bytes")


21 Netflix endpoints
137274 of 141471 packets, 132,110,552 bytes


### Step 2: Counting Traffic to Each Netflix Destination

An important feature for inferring video quality of experience is the throughput of each flow in the video stream. To compute throughput, we need to divide the number of bytes transferred per unit time.

As a first step towards computing that feature, count the number of packets and bytes, in each direction, to each Netflix IP address in the trace.

#### Count the Number of Downstream Bytes and Packets

In [33]:
import re

CLIENT = "192.168.43.72"

# Wireshark replaced many IPs with names. Map each one back to an IP so every
# packet is counted under the Netflix address it belongs to.
name_to_ip = {d: next(iter(ips)) for d, ips in nf_domain_ips.items() if len(ips) == 1}

def to_ip(endpoint):
    if endpoint in name_to_ip:
        return name_to_ip[endpoint]
    m = (re.match(r"ec2-(\d+)-(\d+)-(\d+)-(\d+)\.", str(endpoint))
         or re.match(r"a(\d+)-(\d+)-(\d+)-(\d+)\.deploy\.", str(endpoint)))
    return ".".join(m.groups()) if m else endpoint

ndf = ndf.assign(src_ip=ndf['Source'].map(to_ip), dst_ip=ndf['Destination'].map(to_ip))

# Domain name(s) that resolved to each Netflix IP (an IP can serve several domains).
ip_to_domains = {}
for domain, ips in nf_domain_ips.items():
    for ip in ips:
        ip_to_domains.setdefault(ip, []).append(domain)

def add_domain(df):
    df.insert(0, 'domain', [", ".join(sorted(ip_to_domains.get(ip, []))) for ip in df.index])
    return df

# Downstream: Netflix IP -> client
downstream = (ndf[(ndf['dst_ip'] == CLIENT) & ndf['src_ip'].isin(nf_ips)]
              .groupby('src_ip')['Length']
              .agg(down_packets='count', down_bytes='sum'))
downstream = add_domain(downstream)
downstream


,domain,down_packets,down_bytes
src_ip,,,
198.38.120.130,ipv4-c001-cdg001-ix.1.oca.nflxvideo.net,33,14844
198.38.120.137,occ-0-56-55.1.nflxso.net,101,116255
198.38.120.153,ipv4-c024-cdg001-ix.1.oca.nflxvideo.net,63,39507
198.38.120.162,ipv4-c063-cdg001-ix.1.oca.nflxvideo.net,338,431178
198.38.120.164,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,4873,7138148
198.38.120.166,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,80084,120607242
198.38.120.167,ipv4-c072-cdg001-ix.1.oca.nflxvideo.net,50,19500
23.57.80.120,"assets.nflxext.com, codex.nflxext.com",1005,1332086
34.252.77.54,push.prod.netflix.com,21,4884


#### Count the Number of Upstream Bytes and Packets

In [34]:
# Upstream: client -> Netflix IP
upstream = (ndf[(ndf['src_ip'] == CLIENT) & ndf['dst_ip'].isin(nf_ips)]
            .groupby('dst_ip')['Length']
            .agg(up_packets='count', up_bytes='sum'))
upstream = add_domain(upstream)
display(upstream)

# Combine both directions; Netflix IPs from DNS that never appear in the trace get zeros.
counts = (pd.DataFrame(index=sorted(nf_ips))
          .join(downstream.drop(columns='domain')).join(upstream.drop(columns='domain'))
          .fillna(0).astype(int))
counts.loc['TOTAL'] = counts.sum()
nf_counts = add_domain(counts)
nf_counts.loc['TOTAL', 'domain'] = ''
nf_counts


,domain,up_packets,up_bytes
dst_ip,,,
198.38.120.130,ipv4-c001-cdg001-ix.1.oca.nflxvideo.net,40,3747
198.38.120.137,occ-0-56-55.1.nflxso.net,89,7106
198.38.120.153,ipv4-c024-cdg001-ix.1.oca.nflxvideo.net,75,7140
198.38.120.162,ipv4-c063-cdg001-ix.1.oca.nflxvideo.net,266,23161
198.38.120.164,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,3170,244455
198.38.120.166,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,47902,3357228
198.38.120.167,ipv4-c072-cdg001-ix.1.oca.nflxvideo.net,69,6146
23.57.80.120,"assets.nflxext.com, codex.nflxext.com",834,60463
34.252.77.54,push.prod.netflix.com,25,3753


,domain,down_packets,down_bytes,up_packets,up_bytes
198.38.120.130,ipv4-c001-cdg001-ix.1.oca.nflxvideo.net,33,14844,40,3747
198.38.120.134,ipv4-c005-cdg001-ix.1.oca.nflxvideo.net,0,0,0,0
198.38.120.137,occ-0-56-55.1.nflxso.net,101,116255,89,7106
198.38.120.153,ipv4-c024-cdg001-ix.1.oca.nflxvideo.net,63,39507,75,7140
198.38.120.162,ipv4-c063-cdg001-ix.1.oca.nflxvideo.net,338,431178,266,23161
198.38.120.164,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,4873,7138148,3170,244455
198.38.120.166,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,80084,120607242,47902,3357228
198.38.120.167,ipv4-c072-cdg001-ix.1.oca.nflxvideo.net,50,19500,69,6146
23.57.80.120,"assets.nflxext.com, codex.nflxext.com",1005,1332086,834,60463
34.252.77.54,push.prod.netflix.com,21,4884,25,3753


### Step 3: Inferring Segment Downloads

Another important feature that can be used in inferring video quality of experience is the number of segments per unit time. In this step we will infer the number of segments downloaded per unit time for each IP address.

The number of segments can be determined by counting the number of continuous downstream transfers separated by a packet with a payload of zero bytes. For the last step, compute the number of segment downloads from each Netflix IP address.

In [35]:
# Downstream packets from each Netflix IP, in capture order.
dpk = ndf[(ndf['dst_ip'] == CLIENT) & ndf['src_ip'].isin(nf_ips)]

# The capture is truncated (snaplen 200), so the Info field of data packets has no
# "Len=". A TCP packet with no payload is just headers (at most 78 bytes on the wire:
# 14 Ethernet + 20 IP + 20 TCP + options); every data packet in this trace is larger.
dpk = dpk.assign(zero_payload=dpk['Length'] <= 78)

def count_segments(pkts):
    """A segment is a run of consecutive data packets ended by a zero-payload packet."""
    is_data = ~pkts['zero_payload']
    starts = is_data & ~is_data.shift(fill_value=False)
    return int(starts.sum())

segments = (dpk.groupby('src_ip')
               .apply(count_segments, include_groups=False)
               .rename('segments'))

seg_counts = (pd.DataFrame(index=sorted(nf_ips))
              .join(segments).fillna(0).astype(int))
seg_counts.loc['TOTAL'] = seg_counts.sum()
seg_counts = add_domain(seg_counts)
seg_counts.loc['TOTAL', 'domain'] = ''
seg_counts


,domain,segments
198.38.120.130,ipv4-c001-cdg001-ix.1.oca.nflxvideo.net,2
198.38.120.134,ipv4-c005-cdg001-ix.1.oca.nflxvideo.net,0
198.38.120.137,occ-0-56-55.1.nflxso.net,3
198.38.120.153,ipv4-c024-cdg001-ix.1.oca.nflxvideo.net,5
198.38.120.162,ipv4-c063-cdg001-ix.1.oca.nflxvideo.net,6
198.38.120.164,ipv4-c069-cdg001-ix.1.oca.nflxvideo.net,24
198.38.120.166,ipv4-c071-cdg001-ix.1.oca.nflxvideo.net,136
198.38.120.167,ipv4-c072-cdg001-ix.1.oca.nflxvideo.net,5
23.57.80.120,"assets.nflxext.com, codex.nflxext.com",36
34.252.77.54,push.prod.netflix.com,3


## Thought Question

In this exercise, we used the domain name system (DNS) lookup traffic to identify Netflix traffic. This approach can work in practice but is far from perfect, for a number of reasons:

* Domain names can change over time.
* DNS traffic is becoming increasingly encrypted, making it difficult to see domain name lookups and responses.

This turns the problem of *service identification* (i.e., identifying Netflix traffic itself) into an inference/machine learning problem. What features can you think of that could work for developing a model that can perform this type of inference?